# Dataset Characterization: TMDB vs Airbnb London

Initial characterization EDA for the two candidate datasets used in this
dissertation: the Full TMDB Movies Dataset 2024 and Inside Airbnb London
listings. Findings are logged to `reports/eda_log.md` via
`src/eda/report.py`; figures are saved to `reports/figures/` at 300dpi.

## Setup

### Imports & configuration

In [ ]:
%matplotlib inline

import ast
import json
import re
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import missingno as msno
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from langdetect import LangDetectException, detect
from scipy import stats
from sklearn.preprocessing import MultiLabelBinarizer
from transformers import AutoTokenizer

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.eda import config
from src.eda.loaders import load_airbnb, load_tmdb
from src.eda.report import log_findings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

FIGURES_DIR = REPO_ROOT / "reports" / "figures"
INTERIM_DIR = REPO_ROOT / "data" / "interim"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)


def savefig(fig, filename):
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight")
    print(f"Saved: {path}")

### Reusable helper functions

In [ ]:
def iqr_outlier_count(s):
    """Counts values in `s` that fall outside the 1.5x-IQR Tukey fences.
    This is the standard skew-robust outlier rule, complementary to z-score."""
    s = s.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((s < lower) | (s > upper)).sum())


def zscore_outlier_count(s, threshold=3):
    """Counts values in `s` whose z-score magnitude exceeds `threshold`,
    flagging outliers under a normal-distribution assumption."""
    s = s.dropna()
    if s.std(ddof=0) == 0 or len(s) == 0:
        return 0
    z = stats.zscore(s)
    return int((np.abs(z) > threshold).sum())


def high_corr_pairs(corr, threshold=0.85):
    """Extracts feature pairs from a correlation matrix whose absolute
    correlation exceeds `threshold`, surfacing multicollinearity risks."""
    pairs = []
    cols = corr.columns
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            r = corr.iloc[i, j]
            if pd.notna(r) and abs(r) > threshold:
                pairs.append((cols[i], cols[j], round(r, 3)))
    return pd.DataFrame(pairs, columns=["feature_1", "feature_2", "correlation"])


# DECISION POINT: embedding-dimension heuristic — currently Guo & Berkhahn
# (2016), min(50, cardinality // 2 + 1). If the methodology changes to a
# different heuristic (or a learned/tuned dimension), update here and in the
# "Entity-embedding dimension heuristic" markdown cell below that cites it.
def entity_embedding_dim(cardinality):
    """Suggests an entity-embedding dimension for a categorical feature from
    its cardinality, using the Guo & Berkhahn (2016) heuristic."""
    return min(50, (cardinality // 2) + 1)


def empty_string_report(df, cols):
    """Reports, per column, how many values are null versus present-but-empty
    strings — two distinct missingness signatures that `isnull()` alone conflates."""
    rows = []
    for c in cols:
        as_str = df[c].astype("string")
        rows.append(
            {
                "column": c,
                "null_count": int(df[c].isnull().sum()),
                "empty_string_count": int((as_str.str.strip() == "").sum()),
            }
        )
    return pd.DataFrame(rows)


def text_field_report(df, col, label):
    """Prints and returns the null and empty-string counts for a single
    free-text column."""
    null_count = int(df[col].isnull().sum())
    empty_count = int((df[col].fillna("").astype(str).str.strip() == "").sum())
    print(f"{label} ({col}) — nulls: {null_count}, empty strings: {empty_count}")
    return null_count, empty_count


# DECISION POINT: embedding model choice — currently
# sentence-transformers/all-MiniLM-L6-v2 (512-token max sequence length).
# If the project switches to a different sentence-embedding model, update
# the model name here so token-length stats reflect the model actually used.
TOKENIZER = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")


def token_length_stats(texts, max_len=512, sample_size=5000, random_state=42):
    """Estimates the token-length distribution of a text column under the
    all-MiniLM-L6-v2 tokenizer, from a random sample (tokenizing the full
    column would be prohibitively slow for the TMDB-sized dataset), and
    reports the percentage of sampled rows exceeding `max_len` tokens."""
    texts = texts.dropna().astype(str)
    texts = texts[texts.str.strip() != ""]
    if len(texts) > sample_size:
        texts = texts.sample(sample_size, random_state=random_state)
    lengths = texts.apply(lambda t: len(TOKENIZER.encode(t, truncation=False)))
    pct_over = float((lengths > max_len).mean() * 100)
    return lengths, pct_over


# DECISION POINT: langdetect is a statistical n-gram method that degrades
# badly on very short strings and on text dominated by foreign proper nouns
# (e.g. cast-name lists, short titles) — spot-checking TMDB overview flags
# showed ~5/6 flagged rows were false positives for exactly this reason, not
# genuine non-English text. Currently unfiltered: every flag is treated the
# same regardless of confidence or text length.
# TODO: revisit with either (a) langdetect.detect_langs() + a confidence
# threshold (e.g. >0.9) instead of detect(), or (b) a minimum character-length
# gate that reports very short text as "too short to classify" rather than
# confidently (mis)labeling it, if false positives keep being a problem.
def flag_non_english(series, n=100, random_state=42):
    """Samples a text column and flags entries `langdetect` identifies as
    non-English, for manual spot-checking rather than automatic filtering."""
    sample = series.dropna().astype(str)
    sample = sample[sample.str.strip() != ""]
    sample = sample.sample(min(n, len(sample)), random_state=random_state)

    def safe_detect(t):
        try:
            return detect(t)
        except LangDetectException:
            return "unknown"

    langs = sample.apply(safe_detect)
    non_english = sample[langs != "en"]
    pct = float(len(non_english) / len(sample) * 100) if len(sample) else 0.0
    print(f"{len(non_english)} / {len(sample)} sampled rows ({pct:.1f}%) flagged as non-English")
    return non_english, pct


HTML_RE = re.compile(r"<[^>]+>")


def html_contamination_pct(series):
    """Returns the percentage of a text column's values containing
    HTML/markup tags."""
    texts = series.dropna().astype(str)
    if len(texts) == 0:
        return 0.0
    contaminated = texts.str.contains(HTML_RE)
    return float(contaminated.mean() * 100)


def near_duplicate_report(df, col, id_col="id"):
    """Flags rows whose text, after case/whitespace normalization, exactly
    matches another row's — a simple first-pass near-duplicate check."""
    text = df[col].dropna().astype(str).str.lower().str.strip()
    text = text.str.replace(r"\s+", " ", regex=True)
    text = text[text != ""]  # exclude empty strings (missing, not duplicate)
    dupe_mask = text.duplicated(keep=False)
    n_dupe_rows = int(dupe_mask.sum())
    n_dupe_groups = int(text[dupe_mask].nunique())
    print(f"{n_dupe_rows} rows share duplicated normalized text across {n_dupe_groups} distinct text groups")

    display_cols = [id_col] + (["imdb_id"] if "imdb_id" in df.columns else []) + [col]
    dupe_rows = (
        df.loc[text[dupe_mask].index, display_cols]
        .assign(_normalized=text[dupe_mask])
        .sort_values("_normalized")  # group duplicates together for inspection
        .drop(columns="_normalized")
    )
    return dupe_rows, n_dupe_rows


def parse_list_field(value):
    """Tries JSON / Python-literal list parsing first (e.g. Airbnb amenities);
    falls back to a plain comma split, since TMDB genres/keywords in this
    dataset are stored as comma-separated strings, not JSON."""
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    text = str(value).strip()
    if not text:
        return []
    for parser in (json.loads, ast.literal_eval):
        try:
            parsed = parser(text)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed]
        except (ValueError, SyntaxError):
            continue
    return [item.strip() for item in text.split(",") if item.strip()]


def list_field_summary(df, list_col):
    """Summarizes a parsed list-valued column: vocabulary size, count of
    long-tail items (<10 occurrences), empty-list rate, and per-row list-length
    statistics."""
    lengths = df[list_col].apply(len)
    exploded = df[list_col].explode()
    freq = exploded.value_counts()
    return {
        "vocab_size": int(exploded.nunique()),
        "long_tail_items": int((freq < 10).sum()),
        "pct_empty_lists": float((lengths == 0).mean() * 100),
        "length_describe": lengths.describe(),
        "freq": freq,
    }


# DECISION POINT: multi-hot vs. pooled-embedding cutoff — currently a flat
# vocab_threshold=5000, and the byte estimate assumes 1 byte/cell (uint8
# multi-hot). If this feels too coarse or the multi-hot approach doesn't pan
# out, revisit this threshold and/or evaluate alternatives here.
# TODO: explore pooled SBERT-of-item-names (or hashing-trick / frequency-based
# top-K + "Other" bucketing) as alternatives if multi-hot proves infeasible
# or underperforms for genres/keywords/amenities.
def multihot_feasibility(vocab_size, n_rows, vocab_threshold=5000):
    """Estimates the memory cost of multi-hot encoding a list-valued column
    and warns when the vocabulary is too large for that to be practical."""
    total_bytes = vocab_size * n_rows * 1
    print(
        f"Multi-hot matrix estimate: {vocab_size} vocab x {n_rows} rows "
        f"= {total_bytes:,} bytes (~{total_bytes / 1e6:.1f} MB)"
    )
    if vocab_size > vocab_threshold:
        print(
            "WARNING: vocabulary exceeds 5000 items — multi-hot encoding is "
            "infeasible; use pooled SBERT-of-item-names embeddings instead."
        )
    else:
        print("Multi-hot encoding is feasible at this vocabulary size.")


def top_item_cooccurrence(list_series, top_n=20):
    """Computes a co-occurrence count matrix for the top-N most frequent
    items in a list-valued column, to eyeball natural item clusters."""
    freq = list_series.explode().value_counts()
    top_items = freq.head(top_n).index.tolist()
    restricted = list_series.apply(lambda items: [i for i in items if i in top_items])
    mlb = MultiLabelBinarizer(classes=top_items)
    onehot = mlb.fit_transform(restricted)
    cooc = onehot.T @ onehot
    return pd.DataFrame(cooc, index=top_items, columns=top_items)

### Load datasets

In [ ]:
tmdb_raw = load_tmdb()
airbnb_raw = load_airbnb()

print("TMDB raw shape:", tmdb_raw.shape)
print("Airbnb raw shape:", airbnb_raw.shape)

## 1. Initial checks

### 1.1 Provenance & structural overview

#### Dataset provenance

In [ ]:
display(Markdown(f"""
**Dataset provenance** (pulled from `src/eda/config.py`):

- TMDB download date: `{config.TMDB_DOWNLOAD_DATE}`
- Airbnb scrape date: `{config.AIRBNB_SCRAPE_DATE}`
- Airbnb download date: `{config.AIRBNB_DOWNLOAD_DATE}`
"""))

#### Shape, dtypes & column list

In [ ]:
tmdb_shape_raw = tmdb_raw.shape
airbnb_shape_raw = airbnb_raw.shape
print("TMDB shape:", tmdb_shape_raw)
print("Airbnb shape:", airbnb_shape_raw)

In [ ]:
tmdb_raw.dtypes

In [ ]:
airbnb_raw.dtypes

In [ ]:
print("TMDB columns:")
print(list(tmdb_raw.columns))
print()
print("Airbnb columns:")
print(list(airbnb_raw.columns))

#### Memory usage

In [ ]:
tmdb_mem_mb = tmdb_raw.memory_usage(deep=True).sum() / 1e6
airbnb_mem_mb = airbnb_raw.memory_usage(deep=True).sum() / 1e6
print(f"TMDB memory usage (deep): {tmdb_mem_mb:,.1f} MB")
print(f"Airbnb memory usage (deep): {airbnb_mem_mb:,.1f} MB")

CHUNKED_LOAD_THRESHOLD_MB = 2000
for name, mb in [("TMDB", tmdb_mem_mb), ("Airbnb", airbnb_mem_mb)]:
    if mb > CHUNKED_LOAD_THRESHOLD_MB:
        print(f"FLAG: {name} exceeds {CHUNKED_LOAD_THRESHOLD_MB} MB — consider chunked loading.")
    else:
        print(f"{name} is within comfortable in-memory limits ({mb:,.1f} MB).")

#### Manual sample inspection

In [ ]:
tmdb_raw.sample(10, random_state=42)

In [ ]:
airbnb_raw.sample(10, random_state=42)

#### Missingness

In [ ]:
tmdb_missing = pd.DataFrame(
    {
        "n_missing": tmdb_raw.isnull().sum(),
        "pct_missing": tmdb_raw.isnull().mean() * 100,
    }
).sort_values("pct_missing", ascending=False)
tmdb_missing

In [ ]:
msno.matrix(tmdb_raw.sample(min(len(tmdb_raw), 5000), random_state=42))
plt.title("TMDB missingness pattern (sample)")
fig = plt.gcf()
savefig(fig, "01_tmdb_missingness_matrix.png")
plt.show()

In [ ]:
airbnb_missing = pd.DataFrame(
    {
        "n_missing": airbnb_raw.isnull().sum(),
        "pct_missing": airbnb_raw.isnull().mean() * 100,
    }
).sort_values("pct_missing", ascending=False)
airbnb_missing

In [ ]:
msno.matrix(airbnb_raw.sample(min(len(airbnb_raw), 5000), random_state=42))
plt.title("Airbnb missingness pattern (sample)")
fig = plt.gcf()
savefig(fig, "01_airbnb_missingness_matrix.png")
plt.show()

#### Duplicate `id` / `imdb_id` check

In [ ]:
tmdb_dupe_ids = int(tmdb_raw["id"].duplicated().sum())
airbnb_dupe_ids = int(airbnb_raw["id"].duplicated().sum())
print(f"TMDB duplicate ids: {tmdb_dupe_ids}")
print(f"Airbnb duplicate ids: {airbnb_dupe_ids}")

In [ ]:
airbnb_dupe_id_rows = airbnb_raw[airbnb_raw["id"].duplicated(keep=False)].sort_values("id")
airbnb_dupe_id_rows[["id", "name", "host_id", "price"]]

In [ ]:
# Do duplicate-id rows actually agree on core identity fields, or do they
# genuinely conflict (which would mean the same id was reused across two
# different movies, not just re-entered with more fields filled in)?
key_cols = ["imdb_id", "title", "release_date", "budget", "revenue", "vote_average"]
tmdb_dupe_id_rows = tmdb_raw[tmdb_raw["id"].duplicated(keep=False)]
n_distinct_non_null = tmdb_dupe_id_rows.groupby("id")[key_cols].agg(lambda s: s.dropna().nunique())
conflicting_ids = n_distinct_non_null[(n_distinct_non_null > 1).any(axis=1)]
n_dupe_id_groups = tmdb_dupe_id_rows["id"].nunique()
print(f"{len(conflicting_ids)} of {n_dupe_id_groups} duplicate-id groups "
      f"have conflicting (not just missing-vs-present) values in {key_cols}")

tmdb_dupe_id_rows[tmdb_dupe_id_rows["id"].isin(conflicting_ids.index)][["id"] + key_cols].sort_values("id")

#### `vote_average` / `vote_count` zero combinations

In [ ]:
vote_avg_zero = tmdb_raw["vote_average"] == 0
vote_count_zero = tmdb_raw["vote_count"] == 0

n_both_zero = int((vote_avg_zero & vote_count_zero).sum())
n_avg_nonzero_count_zero = int((~vote_avg_zero & vote_count_zero).sum())
n_avg_zero_count_nonzero = int((vote_avg_zero & ~vote_count_zero).sum())
n_both_nonzero = int((~vote_avg_zero & ~vote_count_zero).sum())

print(f"vote_average==0 and vote_count==0:    {n_both_zero}")
print(f"vote_average!=0 and vote_count==0:    {n_avg_nonzero_count_zero}")
print(f"vote_average==0 and vote_count!=0:    {n_avg_zero_count_nonzero}")
print(f"vote_average!=0 and vote_count!=0:    {n_both_nonzero}")
print(f"total:                                {len(tmdb_raw)}")

# DECISION: vote_count==0 means zero votes were cast, so vote_average cannot
# be anything but 0 in that case -- an average requires at least one value
# to average. The 582 rows above with vote_count==0 but vote_average!=0 are
# therefore logically impossible, not just noisy: corrupted/erroneous records.
# TODO: remove (or recode vote_average to NaN) for these 582 rows as part of
# Section 1.2's cleaning, once the target-variable decision (deferred in
# Section 1.4) is finalized.

In [ ]:
# imdb_id is an external reference, not TMDB's own primary key, so unlike
# id above it can legitimately repeat if the same real movie was entered
# more than once in TMDB — a stronger duplicate-record signal than two rows
# merely sharing a title. "tt0000000" is IMDb's own placeholder for "no ID
# assigned" (not a real shared ID), so it's excluded here to avoid treating
# unrelated movies with an unknown imdb_id as false-positive duplicates.
tmdb_imdb_id_present = tmdb_raw["imdb_id"].dropna()
tmdb_imdb_id_present = tmdb_imdb_id_present[tmdb_imdb_id_present != "tt0000000"]
tmdb_dupe_imdb_id_mask = tmdb_imdb_id_present.duplicated(keep=False)
tmdb_dupe_imdb_ids = int(tmdb_imdb_id_present.duplicated().sum())
print(f"TMDB duplicate imdb_ids (excluding missing/placeholder values): {tmdb_dupe_imdb_ids} "
      f"out of {len(tmdb_imdb_id_present)} non-null, non-placeholder values")

tmdb_dupe_imdb_id_rows = tmdb_raw.loc[
    tmdb_imdb_id_present[tmdb_dupe_imdb_id_mask].index,
    ["id", "imdb_id", "title", "release_date"],
].sort_values("imdb_id")
tmdb_dupe_imdb_id_rows.head(20)

### 1.2 TMDB cleaning: status filter, deduplication & zero-coding

#### Filter to `status == 'Released'`

In [ ]:
status_counts = tmdb_raw["status"].value_counts(dropna=False)
status_pct = (status_counts / len(tmdb_raw) * 100).round(2)
pd.DataFrame({"count": status_counts, "pct": status_pct})

In [ ]:
# Verify the claim behind the filter across every column, not just the
# handful expected to be post-release-only.
released_mask = tmdb_raw["status"] == "Released"
missingness_by_status = pd.DataFrame({
    "% missing (Released)": tmdb_raw.loc[released_mask].isnull().mean() * 100,
    "% missing (non-Released)": tmdb_raw.loc[~released_mask].isnull().mean() * 100,
})
missingness_by_status["gap"] = missingness_by_status["% missing (non-Released)"] - missingness_by_status["% missing (Released)"]
missingness_by_status.round(2).sort_values("gap", ascending=False)

In [ ]:
tmdb = tmdb_raw.copy()
shape_before_status_filter = tmdb.shape
tmdb = tmdb[tmdb["status"] == "Released"].copy()
tmdb_shape_after_status_filter = tmdb.shape
print(f"TMDB shape before status filter: {shape_before_status_filter}")
print(f"TMDB shape after status filter: {tmdb_shape_after_status_filter}")

#### Recode `budget`/`revenue` zeros as NaN

In [ ]:
zero_recode_report = {}
for col in ["budget", "revenue"]:
    pct_zero = float((tmdb[col] == 0).mean() * 100)
    zero_recode_report[col] = pct_zero
    tmdb[col] = tmdb[col].replace(0, np.nan)
    print(f"{col}: {pct_zero:.1f}% of rows were 0 -> recoded as NaN")

#### Recode `runtime` zeros as NaN

In [ ]:
pct_zero_runtime = float((tmdb["runtime"] == 0).mean() * 100)
tmdb["runtime"] = tmdb["runtime"].replace(0, np.nan)
print(f"runtime: {pct_zero_runtime:.1f}% of rows were 0 -> recoded as NaN")

#### Recode placeholder `imdb_id` values as NaN

In [ ]:
tmdb_imdb_id_placeholder_pct = float((tmdb["imdb_id"] == "tt0000000").mean() * 100)
tmdb["imdb_id"] = tmdb["imdb_id"].replace("tt0000000", np.nan)
print(f"imdb_id: {tmdb_imdb_id_placeholder_pct:.3f}% of rows were the placeholder "
      f"'tt0000000' -> recoded as NaN")

#### Drop duplicate `id` rows (keep most complete)

In [ ]:
# DECISION POINT: "most complete" is currently just the row with the
# fewest total NaNs across ALL columns, treating every column as equally
# important. A row missing 2 minor fields (e.g. homepage, poster_path)
# scores the same as one missing 2 consequential ones (e.g. budget,
# revenue). Ties are broken arbitrarily by original row order (stable sort).
# TODO: revisit with column-weighted completeness (e.g. weight budget/
# revenue/overview higher than homepage/poster_path) if this heuristic
# turns out to pick the wrong row in practice.
tmdb_shape_before_dedup = tmdb.shape
tmdb = (
    tmdb.assign(_null_count=tmdb.isnull().sum(axis=1))
    .sort_values("_null_count")
    .drop_duplicates(subset="id", keep="first")
    .drop(columns="_null_count")
    .sort_index()
)
tmdb_shape_after_dedup = tmdb.shape
tmdb_n_dupe_ids_dropped = tmdb_shape_before_dedup[0] - tmdb_shape_after_dedup[0]
print(f"TMDB: dropped {tmdb_n_dupe_ids_dropped} duplicate-id rows (kept the most complete row per id)")
print(f"Shape before dedup: {tmdb_shape_before_dedup}, after: {tmdb_shape_after_dedup}")

### 1.3 Airbnb cleaning: price verification

`load_airbnb()` already strips the currency symbol and converts `price` to
`float64` (see `src/eda/loaders.py`) — re-implementing that parsing here
would duplicate loader logic, so this step verifies the result instead.

#### Verify parsed price & count missing/failed conversions

In [ ]:
airbnb = airbnb_raw.copy()
print("price dtype:", airbnb["price"].dtype)
airbnb_price_missing = int(airbnb["price"].isnull().sum())
print(f"Rows with missing/unparseable price: {airbnb_price_missing} "
      f"({airbnb_price_missing / len(airbnb) * 100:.1f}%)")

### 1.4 Candidate outcome variable distributions

#### TMDB `vote_average` distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
tmdb["vote_average"].dropna().hist(bins=40, ax=ax)
ax.set_title("TMDB: vote_average distribution")
ax.set_xlabel("vote_average")
ax.set_ylabel("count")
savefig(fig, "01_tmdb_vote_average_hist.png")
plt.show()

In [ ]:
for threshold in [1, 5, 10, 20, 50]:
    print(f"vote_count >= {threshold}: {(tmdb['vote_count'] >= threshold).sum():,} rows")

In [ ]:
vote_avg_zero = tmdb_raw["vote_average"] == 0
vote_count_zero = tmdb_raw["vote_count"] == 0

n_both_zero = int((vote_avg_zero & vote_count_zero).sum())
n_avg_nonzero_count_zero = int((~vote_avg_zero & vote_count_zero).sum())
n_avg_zero_count_nonzero = int((vote_avg_zero & ~vote_count_zero).sum())
n_both_nonzero = int((~vote_avg_zero & ~vote_count_zero).sum())

print(f"vote_average==0 and vote_count==0:    {n_both_zero}")
print(f"vote_average!=0 and vote_count==0:    {n_avg_nonzero_count_zero}")
print(f"vote_average==0 and vote_count!=0:    {n_avg_zero_count_nonzero}")
print(f"vote_average!=0 and vote_count!=0:    {n_both_nonzero}")
print(f"total:                                {len(tmdb_raw)}")

# DECISION: vote_count==0 means zero votes were cast, so vote_average cannot
# be anything but 0 in that case -- an average requires at least one value
# to average. The 582 rows above with vote_count==0 but vote_average!=0 are
# therefore logically impossible, not just noisy: corrupted/erroneous records.
# TODO: remove (or recode vote_average to NaN) for these 582 rows as part of
# Section 1.2's cleaning, once the target-variable decision (deferred in
# Section 1.4) is finalized.

#### Airbnb `price` distribution (post-cleaning)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
airbnb["price"].dropna().hist(bins=40, ax=ax)
ax.set_title("Airbnb: price distribution")
ax.set_xlabel("price (GBP)")
ax.set_ylabel("count")
savefig(fig, "01_airbnb_price_hist.png")
plt.show()

## 2. Numerical features (Section 2a)

In [ ]:
TMDB_NUM_COLS = [c for c in ["vote_average", "vote_count", "revenue", "runtime", "budget", "popularity"] if c in tmdb.columns]
TMDB_TARGET = "vote_average"

print("TMDB numeric columns:", TMDB_NUM_COLS)

In [ ]:
AIRBNB_ID_COLS = ["id", "scrape_id", "host_id", "host_profile_id"]
AIRBNB_GEO_COLS = ["latitude", "longitude"]
AIRBNB_ALL_NULL_COLS = airbnb.columns[airbnb.isnull().all()].tolist()
AIRBNB_EMPTY_OR_UNRELIABLE_COLS = [
    "host_since", "host_response_time", "host_thumbnail_url",
    "host_acceptance_rate", "host_response_rate", "host_verifications",
    "neighbourhood", "host_total_listings_count", "host_neighbourhood",
    "calendar_updated", "instant_bookable", "license",
    "neighbourhood_group_cleansed",
]

AIRBNB_NUM_COLS = [
    c for c in airbnb.select_dtypes(include="number").columns
    if c not in AIRBNB_ID_COLS + AIRBNB_GEO_COLS + AIRBNB_ALL_NULL_COLS + AIRBNB_EMPTY_OR_UNRELIABLE_COLS
]
AIRBNB_TARGET = "price"

print("Airbnb numeric columns:", AIRBNB_NUM_COLS)

### Descriptive stats & outliers

#### `describe()` and physically impossible values

In [ ]:
tmdb[TMDB_NUM_COLS].describe()

In [ ]:
airbnb[AIRBNB_NUM_COLS].describe()

In [ ]:
impossible_flags = []
if "runtime" in tmdb.columns:
    impossible_flags.append(("TMDB", "runtime == 0", int((tmdb["runtime"] == 0).sum())))
    impossible_flags.append(("TMDB", "runtime < 0", int((tmdb["runtime"] < 0).sum())))
if "revenue" in tmdb.columns:
    impossible_flags.append(("TMDB", "revenue < 0", int((tmdb["revenue"] < 0).sum())))
if "budget" in tmdb.columns:
    impossible_flags.append(("TMDB", "budget < 0", int((tmdb["budget"] < 0).sum())))
if "price" in airbnb.columns:
    impossible_flags.append(("Airbnb", "price <= 0", int((airbnb["price"] <= 0).sum())))
if "minimum_nights" in airbnb.columns:
    impossible_flags.append(("Airbnb", "minimum_nights <= 0", int((airbnb["minimum_nights"] <= 0).sum())))
if "accommodates" in airbnb.columns:
    impossible_flags.append(("Airbnb", "accommodates <= 0", int((airbnb["accommodates"] <= 0).sum())))

pd.DataFrame(impossible_flags, columns=["dataset", "check", "n_rows"])

In [ ]:
tmdb_neg_recode_report = {}
for col in ["runtime", "revenue"]:
    pct_negative = float((tmdb[col] < 0).mean() * 100)
    tmdb_neg_recode_report[col] = pct_negative
    tmdb[col] = tmdb[col].where(tmdb[col] >= 0, np.nan)
    print(f"{col}: {pct_negative:.4f}% of rows were negative -> recoded as NaN")

#### Outlier detection: IQR (1.5x) vs z-score (|z|>3)

In [ ]:
tmdb_outlier_summary = pd.DataFrame({
    "column": TMDB_NUM_COLS,
    "iqr_outliers": [iqr_outlier_count(tmdb[c]) for c in TMDB_NUM_COLS],
    "zscore_outliers": [zscore_outlier_count(tmdb[c]) for c in TMDB_NUM_COLS],
})
tmdb_outlier_summary["disagreement"] = (
    tmdb_outlier_summary["iqr_outliers"] - tmdb_outlier_summary["zscore_outliers"]
).abs()
tmdb_outlier_summary

In [ ]:
airbnb_outlier_summary = pd.DataFrame({
    "column": AIRBNB_NUM_COLS,
    "iqr_outliers": [iqr_outlier_count(airbnb[c]) for c in AIRBNB_NUM_COLS],
    "zscore_outliers": [zscore_outlier_count(airbnb[c]) for c in AIRBNB_NUM_COLS],
})
airbnb_outlier_summary["disagreement"] = (
    airbnb_outlier_summary["iqr_outliers"] - airbnb_outlier_summary["zscore_outliers"]
).abs()
airbnb_outlier_summary

In [ ]:
redundancy_pairs = [
    ("price", "price_quote_price_per_night"),
    ("price", "price_quote_total_price"),
    ("minimum_nights", "minimum_minimum_nights"),
    ("maximum_nights", "maximum_maximum_nights"),
]
for col_a, col_b in redundancy_pairs:
    pct_equal = float((airbnb[col_a] == airbnb[col_b]).mean() * 100)
    print(f"{col_a} == {col_b}: {pct_equal:.2f}% of rows match")

In [ ]:
hosts_time_cols = [
    "hosts_time_as_user_years", "hosts_time_as_user_months",
    "hosts_time_as_host_years", "hosts_time_as_host_months",
]
airbnb[hosts_time_cols].isnull().mean() * 100

IQR and z-score disagree substantially wherever a column is skewed rather
than roughly normal: IQR is robust to skew and flags points relative to the
central 50%, while z-score assumes a symmetric distribution and under- or
over-counts outliers accordingly. Treat the two counts as complementary,
not interchangeable.

#### Histogram

In [ ]:
fig, axes = plt.subplots(1, len(TMDB_NUM_COLS), figsize=(4 * len(TMDB_NUM_COLS), 5))
for ax, col in zip(np.atleast_1d(axes), TMDB_NUM_COLS):
    values = tmdb[col].dropna()
    values = values[values > 0]
    bins = np.logspace(np.log10(values.min()), np.log10(values.max()), 40)
    ax.hist(values, bins=bins)
    ax.set_xscale("log")
    ax.set_title(col)
fig.suptitle("TMDB: numeric feature histograms (log scale, positive values only)")
savefig(fig, "02_tmdb_histograms.png")
plt.show()

In [ ]:
n_cols = 5
n_rows = int(np.ceil(len(AIRBNB_NUM_COLS) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes = np.atleast_1d(axes).flatten()
for ax, col in zip(axes, AIRBNB_NUM_COLS):
    values = airbnb[col].dropna()
    values = values[values > 0]
    bins = np.logspace(np.log10(values.min()), np.log10(values.max()), 40)
    ax.hist(values, bins=bins)
    ax.set_xscale("log")
    ax.set_title(col, fontsize=9)
for ax in axes[len(AIRBNB_NUM_COLS):]:
    ax.axis("off")
fig.suptitle("Airbnb: numeric feature histograms (log scale, positive values only)")
fig.tight_layout(rect=[0, 0, 1, 0.97])
savefig(fig, "02_airbnb_histograms.png")
plt.show()

#### Boxplots

In [ ]:
fig, axes = plt.subplots(1, len(TMDB_NUM_COLS), figsize=(4 * len(TMDB_NUM_COLS), 5))
for ax, col in zip(np.atleast_1d(axes), TMDB_NUM_COLS):
    tmdb.boxplot(column=col, ax=ax)
    ax.set_title(col)
fig.suptitle("TMDB: numeric feature boxplots")
savefig(fig, "02_tmdb_boxplots.png")
plt.show()

In [ ]:
n_cols = 5
n_rows = int(np.ceil(len(AIRBNB_NUM_COLS) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes = np.atleast_1d(axes).flatten()
for ax, col in zip(axes, AIRBNB_NUM_COLS):
    airbnb.boxplot(column=col, ax=ax)
    ax.set_title(col, fontsize=9)
for ax in axes[len(AIRBNB_NUM_COLS):]:
    ax.axis("off")
fig.suptitle("Airbnb: numeric feature boxplots")
fig.tight_layout(rect=[0, 0, 1, 0.97])
savefig(fig, "02_airbnb_boxplots.png")
plt.show()

In [ ]:
airbnb.nlargest(5, "price")[["id", "name", "price", "property_type", "room_type", "minimum_nights"]]

In [ ]:
airbnb_shape_before_price_drop = airbnb.shape
max_price = airbnb["price"].max()
airbnb = airbnb[airbnb["price"] != max_price]
print(f"Dropped {airbnb_shape_before_price_drop[0] - airbnb.shape[0]} row(s) with price == {max_price}")
print(f"Shape before: {airbnb_shape_before_price_drop}, after: {airbnb.shape}")

In [ ]:
airbnb_shape_before_delisted_drop = airbnb.shape
airbnb = airbnb[airbnb["name"] != "No Longer Available"]
n_delisted_dropped = airbnb_shape_before_delisted_drop[0] - airbnb.shape[0]
print(f"Dropped {n_delisted_dropped} row(s) with name == \"No Longer Available\"")
print(f"Shape before: {airbnb_shape_before_delisted_drop}, after: {airbnb.shape}")

### Skewness & transforms

#### Skewness

In [ ]:
# DECISION POINT: skewness flagging threshold — currently |skew| > 2, the
# more lenient end of common convention (some sources treat |skew| > 1 as
# already "highly skewed"; > 2 follows George & Mallery's more relaxed
# guideline). Columns with skew ~1.0-2.0 won't be flagged at this setting.
# TODO: revisit this threshold (e.g. try 1.0) if downstream modeling shows
# unflagged columns in that 1-2 range are still causing problems.
tmdb_skew = tmdb[TMDB_NUM_COLS].skew().sort_values(ascending=False)
tmdb_skew_flagged = tmdb_skew[tmdb_skew.abs() > 2]
print("TMDB skewness:")
print(tmdb_skew)
print()
print("Flagged (|skew| > 2) as log-transform / quantile-binning candidates:")
print(tmdb_skew_flagged)

In [ ]:
# DECISION POINT: skewness flagging threshold — currently |skew| > 2, the
# more lenient end of common convention (some sources treat |skew| > 1 as
# already "highly skewed"; > 2 follows George & Mallery's more relaxed
# guideline). Columns with skew ~1.0-2.0 won't be flagged at this setting.
# TODO: revisit this threshold (e.g. try 1.0) if downstream modeling shows
# unflagged columns in that 1-2 range are still causing problems.
airbnb_skew = airbnb[AIRBNB_NUM_COLS].skew().sort_values(ascending=False)
airbnb_skew_flagged = airbnb_skew[airbnb_skew.abs() > 2]
print("Airbnb skewness:")
print(airbnb_skew)
print()
print("Flagged (|skew| > 2) as log-transform / quantile-binning candidates:")
print(airbnb_skew_flagged)

#### Airbnb: likely-erroneous high prices (> £2000/night)

In [ ]:
# DECISION POINT: £2000/night flagged as "likely erroneous" — but this is a
# flat threshold with no location context. Genuinely premium areas (e.g.
# Mayfair, Kensington, Notting Hill) may legitimately have listings priced
# above this, so some of these rows could be real, not data errors.
# TODO: cross-check flagged rows against neighbourhood_cleansed / room_type
# before treating any of them as erroneous — investigate whether high prices
# cluster in specific (expensive) locations rather than being random/garbled
# values, and only then decide whether to cap, drop, or leave them as-is.
high_price_rows = airbnb[airbnb["price"] > 2000]
print(f"{len(high_price_rows)} rows with price > £2000/night — shown below, NOT dropped:")
high_price_rows[[c for c in ["id", "name", "price", "room_type", "neighbourhood_cleansed"] if c in high_price_rows.columns]]

### Correlations

#### Correlation matrix & multicollinearity flags

In [ ]:
tmdb_corr = tmdb[TMDB_NUM_COLS].corr()
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(tmdb_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("TMDB: numeric feature correlation matrix")
savefig(fig, "02_tmdb_correlation_heatmap.png")
plt.show()

In [ ]:
tmdb_high_corr_pairs = high_corr_pairs(tmdb_corr, threshold=0.75)
print("TMDB pairs with |r| > 0.75 (multicollinearity risk):")
tmdb_high_corr_pairs

In [ ]:
tmdb_target_corr = tmdb_corr[TMDB_TARGET].drop(TMDB_TARGET).abs().sort_values(ascending=False)
print(f"TMDB numeric features ranked by |r| with {TMDB_TARGET}:")
tmdb_target_corr

In [ ]:
airbnb_corr = airbnb[AIRBNB_NUM_COLS].corr()
n = len(AIRBNB_NUM_COLS)
fig, ax = plt.subplots(figsize=(max(8, n * 0.5), max(7, n * 0.45)))
sns.heatmap(airbnb_corr, annot=False, cmap="coolwarm", center=0, ax=ax)
ax.set_title("Airbnb: numeric feature correlation matrix")
savefig(fig, "02_airbnb_correlation_heatmap.png")
plt.show()
# Too many columns to annotate legibly -- use high_corr_pairs(airbnb_corr) for
# the actual numeric values behind any strong-looking cell above.

In [ ]:
airbnb_high_corr_pairs = high_corr_pairs(airbnb_corr, threshold=0.75)
print("Airbnb pairs with |r| > 0.75 (multicollinearity risk):")
airbnb_high_corr_pairs

In [ ]:
airbnb_target_corr = airbnb_corr[AIRBNB_TARGET].drop(AIRBNB_TARGET).abs().sort_values(ascending=False)
print(f"Airbnb numeric features ranked by |r| with {AIRBNB_TARGET}:")
airbnb_target_corr

## 3. Categorical features (Section 2b)

In [ ]:
# "status" is excluded here: Section 1 already filtered tmdb to
# status == 'Released', so it is now a constant column post-cleaning and
# uninformative for cardinality/long-tail analysis.
TMDB_CAT_COLS = [c for c in ["original_language"] if c in tmdb.columns]
AIRBNB_CAT_COLS = [
    c for c in ["neighbourhood_cleansed", "room_type", "property_type", "host_is_superhost"]
    if c in airbnb.columns
]
print("TMDB categorical columns:", TMDB_CAT_COLS)
print("Airbnb categorical columns:", AIRBNB_CAT_COLS)

### Cardinality & distributions

#### Cardinality

In [ ]:
cardinality_table = pd.concat([
    pd.DataFrame({"dataset": "TMDB", "column": TMDB_CAT_COLS, "cardinality": [tmdb[c].nunique() for c in TMDB_CAT_COLS]}),
    pd.DataFrame({"dataset": "Airbnb", "column": AIRBNB_CAT_COLS, "cardinality": [airbnb[c].nunique() for c in AIRBNB_CAT_COLS]}),
], ignore_index=True)
cardinality_table

#### Value counts & long-tail categories

In [ ]:
for c in TMDB_CAT_COLS:
    vc = tmdb[c].value_counts(normalize=True)
    print(f"--- TMDB: {c} (top 5) ---")
    print(vc.head(5))
    long_tail = vc[vc < 0.01]
    print(f"{len(long_tail)} long-tail categories (<1% share each) — candidates for binning into 'Other'")
    print()

In [ ]:
for c in AIRBNB_CAT_COLS:
    vc = airbnb[c].value_counts(normalize=True)
    print(f"--- Airbnb: {c} (top 5) ---")
    print(vc.head(5))
    long_tail = vc[vc < 0.01]
    print(f"{len(long_tail)} long-tail categories (<1% share each) — candidates for binning into 'Other'")
    print()

#### Long-tail visualization

In [ ]:
CAT_COLS_TO_PLOT = {
    "TMDB": TMDB_CAT_COLS,
    "Airbnb": [c for c in AIRBNB_CAT_COLS if c not in ("host_is_superhost")],
}

for dataset_name, df_, cols in [("TMDB", tmdb, CAT_COLS_TO_PLOT["TMDB"]), ("Airbnb", airbnb, CAT_COLS_TO_PLOT["Airbnb"])]:
    for c in cols:
        vc = df_[c].value_counts(normalize=True)
        top20 = vc.head(20)
        other_share = vc.iloc[20:].sum()
        plot_data = top20.copy()
        if other_share > 0:
            plot_data["Other"] = other_share

        fig, ax = plt.subplots(figsize=(10, 5))
        plot_data.plot(kind="bar", ax=ax)
        ax.set_title(f"{dataset_name}: {c} category frequency (top 20 + Other)")
        ax.set_ylabel("share of rows")
        ax.set_xlabel(c)
        plt.xticks(rotation=45, ha="right")
        fig.tight_layout()
        savefig(fig, f"03_{dataset_name.lower()}_{c}_long_tail.png")
        plt.show()

### Empty-string missingness (distinct from Section 1's `isnull()` check)

In [ ]:
empty_string_report(tmdb, TMDB_CAT_COLS)

In [ ]:
empty_string_report(airbnb, AIRBNB_CAT_COLS)

### Binary-looking field audit

In [ ]:
BINARY_LOOKING_COLS = [c for c in ["host_is_superhost", "instant_bookable", "has_availability"] if c in airbnb.columns]
for c in BINARY_LOOKING_COLS:
    uniques = airbnb[c].unique()
    print(f"{c}: unique values = {uniques}")
    if not set(pd.Series(uniques).dropna()).issubset({True, False}):
        print(f"  FLAG: '{c}' is not a real boolean — likely 't'/'f' strings, needs explicit mapping.")

### Entity-embedding dimension heuristic

Suggested embedding dimension per categorical field, using
`min(50, (cardinality // 2) + 1)` (Guo & Berkhahn, 2016, *Entity Embeddings
of Categorical Variables*).

In [ ]:
def embedding_dim_table(df, cols, dataset_name):
    table = pd.DataFrame({"dataset": dataset_name, "column": cols})
    table["cardinality"] = [df[c].nunique() for c in cols]
    table["suggested_embedding_dim"] = table["cardinality"].apply(entity_embedding_dim)
    return table

embedding_dim_summary = pd.concat([
    embedding_dim_table(tmdb, TMDB_CAT_COLS, "TMDB"),
    embedding_dim_table(airbnb, AIRBNB_CAT_COLS, "Airbnb"),
], ignore_index=True)
embedding_dim_summary

### Fairness-relevant grouping

In [ ]:
FAIRNESS_MIN_GROUP_SIZE = 100

tmdb_lang_counts = tmdb["original_language"].value_counts()
tmdb_small_lang_groups = tmdb_lang_counts[tmdb_lang_counts < FAIRNESS_MIN_GROUP_SIZE]
print(f"TMDB original_language: {len(tmdb_small_lang_groups)} / {len(tmdb_lang_counts)} groups "
      f"have < {FAIRNESS_MIN_GROUP_SIZE} samples — unreliable for later fairness metrics:")
tmdb_small_lang_groups

In [ ]:
airbnb_neigh_counts = airbnb["neighbourhood_cleansed"].value_counts()
airbnb_small_neigh_groups = airbnb_neigh_counts[airbnb_neigh_counts < FAIRNESS_MIN_GROUP_SIZE]
print(f"Airbnb neighbourhood_cleansed: {len(airbnb_small_neigh_groups)} / {len(airbnb_neigh_counts)} groups "
      f"have < {FAIRNESS_MIN_GROUP_SIZE} samples — unreliable for later fairness metrics:")
airbnb_small_neigh_groups

## 4. High-cardinality string / text features (Section 2c)

Covers TMDB `overview` and Airbnb `description` — the long free-text field
in each dataset.

### High-cardinality field audit

Scans every string-typed column (not just `overview`/`description`/`tagline`) for its uniqueness ratio (`nunique / non-null count`), to check whether any other fields deserve the same free-text treatment. List-valued raw string columns with real data (`genres`, `keywords`, `production_companies`, `production_countries`, `spoken_languages`, `amenities`) are excluded here since they are handled separately in Section 5 -- their raw combined-string form isn't the right lens for cardinality. `host_verifications` is also excluded, but for a different reason: it is a list-valued field in Inside Airbnb's schema, but 100% missing in this data pull, so there is nothing to parse or analyze for it.

A high ratio (close to 1.0) doesn't automatically mean a column needs the full text-quality treatment: identifier/URL fields (`imdb_id`, `homepage`, `poster_path`, `backdrop_path` for TMDB; `listing_url`, `picture_url`, `host_url`, `host_thumbnail_url`, `host_picture_url` for Airbnb) will also show high uniqueness but aren't natural language, so they're not worth token-length/language-detection/HTML checks. The genuine candidates to look for here are other short-to-medium narrative text fields, e.g. `title`/`original_title` (TMDB) or `name`/`host_about` (Airbnb) -- both now covered below. (`neighborhood_overview` was also a nominal candidate but is 100% missing in this data pull, so there is nothing to analyze for it either.)

In [ ]:
LIST_VALUED_RAW_COLS = {
    "TMDB": ["genres", "keywords", "production_companies", "production_countries", "spoken_languages"],
    "Airbnb": ["amenities", "host_verifications"],
}

def high_cardinality_scan(df, dataset_name):
    exclude = LIST_VALUED_RAW_COLS.get(dataset_name, [])
    rows = []
    for c in df.columns:
        if c in exclude:
            continue
        if pd.api.types.is_string_dtype(df[c]) or df[c].dtype == object:
            non_null = df[c].notna().sum()
            nunique = df[c].nunique()
            ratio = nunique / non_null if non_null else 0.0
            rows.append((c, nunique, non_null, ratio))
    return pd.DataFrame(
        rows, columns=["column", "nunique", "non_null_count", "uniqueness_ratio"]
    ).sort_values("uniqueness_ratio", ascending=False)

tmdb_cardinality_scan = high_cardinality_scan(tmdb, "TMDB")
print("TMDB string columns by uniqueness ratio:")
tmdb_cardinality_scan

In [ ]:
airbnb_cardinality_scan = high_cardinality_scan(airbnb, "Airbnb")
print("Airbnb string columns by uniqueness ratio:")
airbnb_cardinality_scan

### TMDB

#### `overview`

##### Nulls & empty strings

In [ ]:
tmdb_overview_nulls, tmdb_overview_empty = text_field_report(tmdb, "overview", "TMDB overview")

##### Character length distribution

In [ ]:
tmdb_overview_len = tmdb["overview"].dropna().astype(str).str.len()
tmdb_overview_len.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
tmdb_overview_len.hist(bins=50, ax=ax)
ax.set_title("TMDB overview: character length distribution")
ax.set_xlabel("characters")
savefig(fig, "04_tmdb_overview_length_hist.png")
plt.show()

##### Token length (all-MiniLM-L6-v2 tokenizer, 5000-row sample)

In [ ]:
tmdb_overview_token_lengths, tmdb_overview_pct_over_512 = token_length_stats(tmdb["overview"])
print(f"TMDB overview: {tmdb_overview_pct_over_512:.1f}% of sampled rows exceed 512 tokens")
tmdb_overview_token_lengths.describe()

##### Non-English sample flagging

In [ ]:
tmdb_overview_non_english, tmdb_overview_non_english_pct = flag_non_english(tmdb["overview"])
tmdb_overview_non_english.head(20)

##### HTML/markup contamination

In [ ]:
tmdb_overview_html_pct = html_contamination_pct(tmdb["overview"])
print(f"TMDB overview: {tmdb_overview_html_pct:.2f}% of rows contain HTML/markup")

##### Near-duplicate detection

In [ ]:
tmdb_overview_dupes, tmdb_overview_n_dupe_rows = near_duplicate_report(tmdb, "overview", id_col="id")
tmdb_overview_dupes.head(20)

#### `tagline`

##### Nulls & empty strings

In [ ]:
tmdb_tagline_nulls, tmdb_tagline_empty = text_field_report(tmdb, "tagline", "TMDB tagline")

##### Character length distribution

In [ ]:
tmdb_tagline_len = tmdb["tagline"].dropna().astype(str).str.len()
tmdb_tagline_len.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
tmdb_tagline_len.hist(bins=50, ax=ax)
ax.set_title("TMDB tagline: character length distribution")
ax.set_xlabel("characters")
savefig(fig, "04_tmdb_tagline_length_hist.png")
plt.show()

##### Token length (all-MiniLM-L6-v2 tokenizer, 5000-row sample)

In [ ]:
tmdb_tagline_token_lengths, tmdb_tagline_pct_over_512 = token_length_stats(tmdb["tagline"])
print(f"TMDB tagline: {tmdb_tagline_pct_over_512:.1f}% of sampled rows exceed 512 tokens")
tmdb_tagline_token_lengths.describe()

##### Non-English sample flagging

In [ ]:
tmdb_tagline_non_english, tmdb_tagline_non_english_pct = flag_non_english(tmdb["tagline"])
tmdb_tagline_non_english.head(20)

##### HTML/markup contamination

In [ ]:
tmdb_tagline_html_pct = html_contamination_pct(tmdb["tagline"])
print(f"TMDB tagline: {tmdb_tagline_html_pct:.2f}% of rows contain HTML/markup")

##### Near-duplicate detection

In [ ]:
tmdb_tagline_dupes, tmdb_tagline_n_dupe_rows = near_duplicate_report(tmdb, "tagline", id_col="id")
tmdb_tagline_dupes.head(20)

#### `title`

##### Nulls & empty strings

In [ ]:
tmdb_title_nulls, tmdb_title_empty = text_field_report(tmdb, "title", "TMDB title")

##### Character length distribution

In [ ]:
tmdb_title_len = tmdb["title"].dropna().astype(str).str.len()
tmdb_title_len.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
tmdb_title_len.hist(bins=50, ax=ax)
ax.set_title("TMDB title: character length distribution")
ax.set_xlabel("characters")
savefig(fig, "04_tmdb_title_length_hist.png")
plt.show()

##### Token length (all-MiniLM-L6-v2 tokenizer, 5000-row sample)

In [ ]:
tmdb_title_token_lengths, tmdb_title_pct_over_512 = token_length_stats(tmdb["title"])
print(f"TMDB title: {tmdb_title_pct_over_512:.1f}% of sampled rows exceed 512 tokens")
tmdb_title_token_lengths.describe()

##### Non-English sample flagging

In [ ]:
tmdb_title_non_english, tmdb_title_non_english_pct = flag_non_english(tmdb["title"])
tmdb_title_non_english.head(20)

##### HTML/markup contamination

In [ ]:
tmdb_title_html_pct = html_contamination_pct(tmdb["title"])
print(f"TMDB title: {tmdb_title_html_pct:.2f}% of rows contain HTML/markup")

##### Near-duplicate detection

In [ ]:
tmdb_title_dupes, tmdb_title_n_dupe_rows = near_duplicate_report(tmdb, "title", id_col="id")
tmdb_title_dupes.head(20)

#### `original_title`

##### Nulls & empty strings

In [ ]:
tmdb_original_title_nulls, tmdb_original_title_empty = text_field_report(tmdb, "original_title", "TMDB original_title")

##### Character length distribution

In [ ]:
tmdb_original_title_len = tmdb["original_title"].dropna().astype(str).str.len()
tmdb_original_title_len.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
tmdb_original_title_len.hist(bins=50, ax=ax)
ax.set_title("TMDB original_title: character length distribution")
ax.set_xlabel("characters")
savefig(fig, "04_tmdb_original_title_length_hist.png")
plt.show()

##### Token length (all-MiniLM-L6-v2 tokenizer, 5000-row sample)

In [ ]:
tmdb_original_title_token_lengths, tmdb_original_title_pct_over_512 = token_length_stats(tmdb["original_title"])
print(f"TMDB original_title: {tmdb_original_title_pct_over_512:.1f}% of sampled rows exceed 512 tokens")
tmdb_original_title_token_lengths.describe()

##### Non-English sample flagging

In [ ]:
tmdb_original_title_non_english, tmdb_original_title_non_english_pct = flag_non_english(tmdb["original_title"])
tmdb_original_title_non_english.head(20)

##### HTML/markup contamination

In [ ]:
tmdb_original_title_html_pct = html_contamination_pct(tmdb["original_title"])
print(f"TMDB original_title: {tmdb_original_title_html_pct:.2f}% of rows contain HTML/markup")

##### Near-duplicate detection

In [ ]:
tmdb_original_title_dupes, tmdb_original_title_n_dupe_rows = near_duplicate_report(tmdb, "original_title", id_col="id")
tmdb_original_title_dupes.head(20)

### Airbnb

#### `description`

##### Nulls & empty strings

In [ ]:
airbnb_description_nulls, airbnb_description_empty = text_field_report(airbnb, "description", "Airbnb description")

##### Character length distribution

In [ ]:
airbnb_description_len = airbnb["description"].dropna().astype(str).str.len()
airbnb_description_len.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
airbnb_description_len.hist(bins=50, ax=ax)
ax.set_title("Airbnb description: character length distribution")
ax.set_xlabel("characters")
savefig(fig, "04_airbnb_description_length_hist.png")
plt.show()

##### Token length (all-MiniLM-L6-v2 tokenizer, 5000-row sample)

In [ ]:
airbnb_description_token_lengths, airbnb_description_pct_over_512 = token_length_stats(airbnb["description"])
print(f"Airbnb description: {airbnb_description_pct_over_512:.1f}% of sampled rows exceed 512 tokens")
airbnb_description_token_lengths.describe()

##### Non-English sample flagging

In [ ]:
airbnb_description_non_english, airbnb_description_non_english_pct = flag_non_english(airbnb["description"])
airbnb_description_non_english.head(20)

##### HTML/markup contamination

In [ ]:
airbnb_description_html_pct = html_contamination_pct(airbnb["description"])
print(f"Airbnb description: {airbnb_description_html_pct:.2f}% of rows contain HTML/markup")

##### Near-duplicate detection

In [ ]:
airbnb_description_dupes, airbnb_description_n_dupe_rows = near_duplicate_report(airbnb, "description", id_col="id")
airbnb_description_dupes.head(20)

#### `name`

##### Nulls & empty strings

In [ ]:
airbnb_name_nulls, airbnb_name_empty = text_field_report(airbnb, "name", "Airbnb name")

##### Character length distribution

In [ ]:
airbnb_name_len = airbnb["name"].dropna().astype(str).str.len()
airbnb_name_len.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
airbnb_name_len.hist(bins=50, ax=ax)
ax.set_title("Airbnb name: character length distribution")
ax.set_xlabel("characters")
savefig(fig, "04_airbnb_name_length_hist.png")
plt.show()

##### Token length (all-MiniLM-L6-v2 tokenizer, 5000-row sample)

In [ ]:
airbnb_name_token_lengths, airbnb_name_pct_over_512 = token_length_stats(airbnb["name"])
print(f"Airbnb name: {airbnb_name_pct_over_512:.1f}% of sampled rows exceed 512 tokens")
airbnb_name_token_lengths.describe()

##### Non-English sample flagging

In [ ]:
airbnb_name_non_english, airbnb_name_non_english_pct = flag_non_english(airbnb["name"])
airbnb_name_non_english.head(20)

##### HTML/markup contamination

In [ ]:
airbnb_name_html_pct = html_contamination_pct(airbnb["name"])
print(f"Airbnb name: {airbnb_name_html_pct:.2f}% of rows contain HTML/markup")

##### Near-duplicate detection

In [ ]:
airbnb_name_dupes, airbnb_name_n_dupe_rows = near_duplicate_report(airbnb, "name", id_col="id")
airbnb_name_dupes.head(20)

#### `host_about`

##### Nulls & empty strings

In [ ]:
airbnb_host_about_nulls, airbnb_host_about_empty = text_field_report(airbnb, "host_about", "Airbnb host_about")

##### Character length distribution

In [ ]:
airbnb_host_about_len = airbnb["host_about"].dropna().astype(str).str.len()
airbnb_host_about_len.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
airbnb_host_about_len.hist(bins=50, ax=ax)
ax.set_title("Airbnb host_about: character length distribution")
ax.set_xlabel("characters")
savefig(fig, "04_airbnb_host_about_length_hist.png")
plt.show()

##### Token length (all-MiniLM-L6-v2 tokenizer, 5000-row sample)

In [ ]:
airbnb_host_about_token_lengths, airbnb_host_about_pct_over_512 = token_length_stats(airbnb["host_about"])
print(f"Airbnb host_about: {airbnb_host_about_pct_over_512:.1f}% of sampled rows exceed 512 tokens")
airbnb_host_about_token_lengths.describe()

##### Non-English sample flagging

In [ ]:
airbnb_host_about_non_english, airbnb_host_about_non_english_pct = flag_non_english(airbnb["host_about"])
airbnb_host_about_non_english.head(20)

##### HTML/markup contamination

In [ ]:
airbnb_host_about_html_pct = html_contamination_pct(airbnb["host_about"])
print(f"Airbnb host_about: {airbnb_host_about_html_pct:.2f}% of rows contain HTML/markup")

##### Near-duplicate detection

In [ ]:
airbnb_host_about_dupes, airbnb_host_about_n_dupe_rows = near_duplicate_report(airbnb, "host_about", id_col="id")
airbnb_host_about_dupes.head(20)

## 5. List-valued features (Section 2d)

Covers TMDB `genres`, `keywords`, `production_companies`,
`production_countries`, `spoken_languages`, and Airbnb `amenities`. TMDB
stores these as comma-separated plain strings rather than JSON, so
`parse_list_field()` (defined in Setup) tries `json.loads` /
`ast.literal_eval` first — for Airbnb's JSON-style `amenities` — and falls
back to a comma split otherwise.

### Parsing verification

In [ ]:
for col, df_ in [
    ("genres", tmdb),
    ("keywords", tmdb),
    ("production_companies", tmdb),
    ("production_countries", tmdb),
    ("spoken_languages", tmdb),
    ("amenities", airbnb),
]:
    sample_val = df_[col].dropna().iloc[0]
    print(f"--- {col} ---")
    print("raw:", sample_val)
    print("parsed:", parse_list_field(sample_val))
    print()

In [ ]:
tmdb["genres_list"] = tmdb["genres"].apply(parse_list_field)
tmdb["keywords_list"] = tmdb["keywords"].apply(parse_list_field)
tmdb["production_companies_list"] = tmdb["production_companies"].apply(parse_list_field)
tmdb["production_countries_list"] = tmdb["production_countries"].apply(parse_list_field)
tmdb["spoken_languages_list"] = tmdb["spoken_languages"].apply(parse_list_field)
airbnb["amenities_list"] = airbnb["amenities"].apply(parse_list_field)

### TMDB `genres`

#### Vocabulary & frequency

In [ ]:
tmdb_genres_summary = list_field_summary(tmdb, "genres_list")
tmdb_genres_vocab = tmdb_genres_summary["vocab_size"]
print(f"Vocabulary size: {tmdb_genres_vocab}")
print(f"Long-tail items (<10 occurrences): {tmdb_genres_summary['long_tail_items']}")
tmdb_genres_summary["freq"].head(20)

#### List length & emptiness

In [ ]:
print(tmdb_genres_summary["length_describe"])
print(f"% rows with empty list: {tmdb_genres_summary['pct_empty_lists']:.2f}%")

#### Multi-hot feasibility

In [ ]:
multihot_feasibility(tmdb_genres_vocab, len(tmdb))

#### Co-occurrence of top 20 genres

In [ ]:
top_item_cooccurrence(tmdb["genres_list"], top_n=20)

### TMDB `keywords`

#### Vocabulary & frequency

In [ ]:
tmdb_keywords_summary = list_field_summary(tmdb, "keywords_list")
tmdb_keywords_vocab = tmdb_keywords_summary["vocab_size"]
print(f"Vocabulary size: {tmdb_keywords_vocab}")
print(f"Long-tail items (<10 occurrences): {tmdb_keywords_summary['long_tail_items']}")
tmdb_keywords_summary["freq"].head(20)

#### List length & emptiness

In [ ]:
# DECISION POINT: ~75% of rows have an empty keywords list. "Empty" here
# doesn't yet distinguish genuinely-missing (NaN, never tagged) from
# explicitly-empty (parsed to [] from an empty string) — same disguised-
# missingness risk seen elsewhere in this dataset.
# TODO: check whether emptiness correlates with release_date/popularity
# (structured vs. random), and decide how encoding should represent "no
# keywords" before treating this field as a primary feature.
print(tmdb_keywords_summary["length_describe"])
print(f"% rows with empty list: {tmdb_keywords_summary['pct_empty_lists']:.2f}%")

#### Multi-hot feasibility

In [ ]:
multihot_feasibility(tmdb_keywords_vocab, len(tmdb))

#### Co-occurrence of top 20 keywords

In [ ]:
top_item_cooccurrence(tmdb["keywords_list"], top_n=20)

### TMDB `production_companies`

#### Vocabulary & frequency

In [ ]:
tmdb_production_companies_summary = list_field_summary(tmdb, "production_companies_list")
tmdb_production_companies_vocab = tmdb_production_companies_summary["vocab_size"]
print(f"Vocabulary size: {tmdb_production_companies_vocab}")
print(f"Long-tail items (<10 occurrences): {tmdb_production_companies_summary['long_tail_items']}")
tmdb_production_companies_summary["freq"].head(20)

#### List length & emptiness

In [ ]:
print(tmdb_production_companies_summary["length_describe"])
print(f"% rows with empty list: {tmdb_production_companies_summary['pct_empty_lists']:.2f}%")

#### Multi-hot feasibility

In [ ]:
multihot_feasibility(tmdb_production_companies_vocab, len(tmdb))

#### Co-occurrence of top 20 production companies

In [ ]:
top_item_cooccurrence(tmdb["production_companies_list"], top_n=20)

### TMDB `production_countries`

#### Vocabulary & frequency

In [ ]:
tmdb_production_countries_summary = list_field_summary(tmdb, "production_countries_list")
tmdb_production_countries_vocab = tmdb_production_countries_summary["vocab_size"]
print(f"Vocabulary size: {tmdb_production_countries_vocab}")
print(f"Long-tail items (<10 occurrences): {tmdb_production_countries_summary['long_tail_items']}")
tmdb_production_countries_summary["freq"].head(20)

#### List length & emptiness

In [ ]:
print(tmdb_production_countries_summary["length_describe"])
print(f"% rows with empty list: {tmdb_production_countries_summary['pct_empty_lists']:.2f}%")

#### Multi-hot feasibility

In [ ]:
multihot_feasibility(tmdb_production_countries_vocab, len(tmdb))

#### Co-occurrence of top 20 production countries

In [ ]:
top_item_cooccurrence(tmdb["production_countries_list"], top_n=20)

### TMDB `spoken_languages`

#### Vocabulary & frequency

In [ ]:
tmdb_spoken_languages_summary = list_field_summary(tmdb, "spoken_languages_list")
tmdb_spoken_languages_vocab = tmdb_spoken_languages_summary["vocab_size"]
print(f"Vocabulary size: {tmdb_spoken_languages_vocab}")
print(f"Long-tail items (<10 occurrences): {tmdb_spoken_languages_summary['long_tail_items']}")
tmdb_spoken_languages_summary["freq"].head(20)

#### List length & emptiness

In [ ]:
print(tmdb_spoken_languages_summary["length_describe"])
print(f"% rows with empty list: {tmdb_spoken_languages_summary['pct_empty_lists']:.2f}%")

#### Multi-hot feasibility

In [ ]:
multihot_feasibility(tmdb_spoken_languages_vocab, len(tmdb))

#### Co-occurrence of top 20 spoken languages

In [ ]:
top_item_cooccurrence(tmdb["spoken_languages_list"], top_n=20)

### Airbnb `amenities`

#### Vocabulary & frequency

In [ ]:
airbnb_amenities_summary = list_field_summary(airbnb, "amenities_list")
airbnb_amenities_vocab = airbnb_amenities_summary["vocab_size"]
print(f"Vocabulary size: {airbnb_amenities_vocab}")
print(f"Long-tail items (<10 occurrences): {airbnb_amenities_summary['long_tail_items']}")
airbnb_amenities_summary["freq"].head(20)

#### List length & emptiness

In [ ]:
print(airbnb_amenities_summary["length_describe"])
print(f"% rows with empty list: {airbnb_amenities_summary['pct_empty_lists']:.2f}%")

#### Multi-hot feasibility

In [ ]:
multihot_feasibility(airbnb_amenities_vocab, len(airbnb))

#### Co-occurrence of top 20 amenities

In [ ]:
top_item_cooccurrence(airbnb["amenities_list"], top_n=20)

## Save Stage 1 output

Saves this notebook's TMDB/Airbnb DataFrames as `tmdb_stage1.parquet`/
`airbnb_stage1.parquet` (post status-filter, id-based duplicate removal,
imdb_id placeholder recode, budget/revenue/runtime zero-and-negative
recode, and the parsed list-valued columns) so later notebooks can
start from here without re-running this full load-and-clean sequence.

**This is not yet the analysis-ready file** -- known issues remain
unresolved at this point and get fixed in Stage 2:
- imdb_id-based duplicates (distinct `id` rows sharing the same
  imdb_id) are flagged in Section 1.1 but not yet deduplicated
- corrupted vote_average/vote_count rows (582 rows with
  vote_count==0 but vote_average!=0) are flagged but not yet
  recoded/dropped
- `flag_non_english`'s high false-positive rate on short text is
  documented but not corrected

Note: `vote_average`'s own zero-inflation (`vote_count == 0`) is
intentionally NOT recoded here yet, pending a decision on the target
variable (see the discussion around Section 1.4).

In [ ]:
tmdb_interim_path = INTERIM_DIR / "tmdb_stage1.parquet"
airbnb_interim_path = INTERIM_DIR / "airbnb_stage1.parquet"

tmdb.to_parquet(tmdb_interim_path, index=False)
airbnb.to_parquet(airbnb_interim_path, index=False)

print(f"Saved: {tmdb_interim_path} ({tmdb.shape})")
print(f"Saved: {airbnb_interim_path} ({airbnb.shape})")

In [ ]:
tmdb.head(2)

In [ ]:
airbnb.head(2)

## Log combined summary

In [ ]:
findings = {
    "TMDB rows (raw -> released, zero-coded)": f"{tmdb_shape_raw[0]} -> {tmdb.shape[0]}",
    "Airbnb rows (raw -> cleaned)": f"{airbnb_shape_raw[0]} -> {airbnb.shape[0]}",
    "TMDB memory footprint (deep)": f"{tmdb_mem_mb:,.1f} MB",
    "Airbnb memory footprint (deep)": f"{airbnb_mem_mb:,.1f} MB",
    "TMDB most predictive numeric features (|r| with vote_average)": tmdb_target_corr.head(3).round(3).to_dict(),
    "Entity-embedding dimension table (see notebook Section 3)": embedding_dim_summary.set_index(["dataset", "column"])["suggested_embedding_dim"].to_dict(),
    "TMDB overview: % > 512 tokens (sampled)": f"{tmdb_overview_pct_over_512:.1f}%",
    "Airbnb description: % > 512 tokens (sampled)": f"{airbnb_description_pct_over_512:.1f}%",
    "TMDB overview: % flagged non-English (sampled)": f"{tmdb_overview_non_english_pct:.1f}%",
    "Airbnb description: % flagged non-English (sampled)": f"{airbnb_description_non_english_pct:.1f}%",
    "TMDB overview: % HTML-contaminated": f"{tmdb_overview_html_pct:.2f}%",
    "Airbnb description: % HTML-contaminated": f"{airbnb_description_html_pct:.2f}%",
    "TMDB genres vocabulary size": tmdb_genres_vocab,
    "TMDB keywords vocabulary size": tmdb_keywords_vocab,
    "Airbnb amenities vocabulary size": airbnb_amenities_vocab,
    "Multi-hot vs pooled-embedding recommendation": (
        "genres: multi-hot feasible; keywords/amenities: use pooled "
        "SBERT-of-item-names if vocab_size > 5000 (see Section 5 warnings above)"
    ),
}

log_findings(
    title="01 Characterization: TMDB vs Airbnb — combined summary",
    findings=findings,
    figures=[
        "reports/figures/01_tmdb_vote_average_hist.png",
        "reports/figures/01_airbnb_price_hist.png",
        "reports/figures/02_tmdb_boxplots.png",
        "reports/figures/02_airbnb_boxplots.png",
        "reports/figures/02_tmdb_correlation_heatmap.png",
        "reports/figures/02_airbnb_correlation_heatmap.png",
        "reports/figures/04_tmdb_overview_length_hist.png",
        "reports/figures/04_airbnb_description_length_hist.png",
    ],
)
print("Logged combined summary to reports/eda_log.md")